#<h1 align="center">**LMF Interactive PCA**</h1>




<div align="justify">

This is an interactive UMAP plot where you can easily explore all the processed samples from the Low Methane Forages project.

First go to **File** → **"Save a copy in Drive"**. This will create a copy of the notebook in your Google Drive. You can then edit the notebook and explore the data using the interactive controls.

To activate the visualization options, click the run_button.jpg button located at the top of the panel. Then, explore the different  functional groups. Once you have adjusted the plot to your preference, you can export it by clicking the three_dots_button.jpg button in the upper-right corner of the scatter plot.

**Advanced options**: You can customize the benchmark visualization by adding one or more id_lab values to the list in the second cell of the **Plot code** section. Benchmark samples will be highlighted in **purple**.

To highlight one or more samples of interest, add their **id** values to the target list. These samples will be displayed in **red**.

Don't know the id of your sample of interest? Explore the complete dataset [here.](https://github.com/maurope/lmf/blob/main/data/2026_08_20_database_categories_quartiles__visualization_public/compiled_categories_quartiles.csv)

</div>


# 1.0 Import libraries

In [1]:
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from IPython.display import display

#2.0 Data load

In [2]:
url = "https://raw.githubusercontent.com/maurope/lmf/main/data/2026_08_20_database_categories_quartiles__visualization_public/compiled_categories_quartiles.csv"
df = pd.read_csv(url)
df

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,n_replicates_nutrition,dm_percentage,ash_dm,...,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,F24-3470,CIAT-11194,1,199.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,92.89,9.61,...,15.17,18.02,46.12,59.25,Medium,Not High,Category_2,81.0,Q1,58.0
1,F24-3471,CIAT-11999,1,201.0,Genetic_bank,Stylosanthes guianensis,Herbaceous_legumes,2,93.50,10.46,...,12.98,16.25,46.61,58.55,Medium,Not High,Category_2,88.0,Q1,66.0
2,F24-3472,CIAT-12318,1,203.0,Genetic_bank,Stylosanthes hamata,Herbaceous_legumes,2,94.10,11.08,...,13.99,16.96,52.27,56.88,Medium,Not High,Category_2,148.0,Q2,78.0
3,F24-3427,CIAT-1257,1,113.0,Genetic_bank,Stylosanthes scabra,Herbaceous_legumes,2,92.72,9.52,...,15.71,17.60,46.32,58.95,Medium,Not High,Category_2,84.0,Q1,62.0
4,F24-3473,CIAT-13575,1,205.0,Genetic_bank,Desmodium incanum,Herbaceous_legumes,2,94.14,11.52,...,13.60,16.23,42.43,42.85,Low,Not High,Category_2,46.0,Q3,224.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Medium,Not High,Category_2,151.0,NaN,NaN
689,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,1.0,NaN,NaN
690,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,High,Not High,Category_4,36.0,NaN,NaN
691,F25-2641,NaN,4,63.0,Mauricio Sotelo,Gliricidia sepium,Shrub_Trees,2,93.50,9.41,...,13.69,14.98,45.48,55.91,Low,Not High,Category_2,55.0,NaN,NaN


# 3.0 PCA code

In [3]:
# ============================================================
# Lists of special samples
# ============================================================

target = ["CIAT-PM-21-2580"]

star_grass_control_ids = [
    "Sample-8"
]

benchmark_ids = [
    "CIAT-6962-Mombaza-Ex",
    "CIAT-606-Basilisk-Opt",
    "CIAT-6294-Marandu-Opt",
    "CIAT-36087-MulatoII-Opt",
    "CIAT-6133-Llanero-Opt",
    "CIAT-6962-Mombaza-Opt",
    "Paja-Sabana-Madura",
    "Humidicola-679",
    "Sabana-Quemada",
    "CIAT-6294-Marandu-Exc",
    "CIAT-606-Basilisk-Exc",
    "BR-02-1752-Cayman-Exc",
    "BR-06-423-Cayman-Exc"
]

In [9]:
# ============================================================
# PCA variables
# ============================================================

pca_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]


# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "point_category": "Plot category",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}


# ============================================================
# Tooltip columns
# ============================================================

tooltip_columns = [
    "id",
    "id_lab",
    "tax_name",
    "functional_group",
    "point_category",
    "subset",
    "n_replicates_nutrition",
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "n_replicates_gas",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm",
    "ch4_category",
    "tddm_category",
    "lmf_category",
    "lmf_category_rank",
    "quartile",
    "quartile_rank"
]


# ============================================================
# Check PCA variables
# ============================================================

missing_pca_vars = [
    column
    for column in pca_vars
    if column not in df.columns
]

if missing_pca_vars:

    raise ValueError(
        "The following PCA variables are missing from df: "
        f"{missing_pca_vars}"
    )


# ============================================================
# Clean functional group
# ============================================================

df["functional_group"] = (
    df["functional_group"]
    .astype("string")
    .str.strip()
)


# ============================================================
# Valid functional groups
# ============================================================

valid_functional_groups = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees"
]


# ============================================================
# Filter data for PCA
# ============================================================

df_pca = df[
    df["functional_group"].isin(
        valid_functional_groups
    )
].copy()


# ============================================================
# Special sample categories
# ============================================================

# Default category = functional group
df_pca["point_category"] = (
    df_pca["functional_group"]
)


# ------------------------------------------------------------
# Benchmark samples
# ------------------------------------------------------------

df_pca.loc[
    df_pca["id"].isin(benchmark_ids),
    "point_category"
] = "Benchmark"


# ------------------------------------------------------------
# Star grass control
# ------------------------------------------------------------

df_pca.loc[
    df_pca["id"].isin(star_grass_control_ids),
    "point_category"
] = "Star grass control"


# ------------------------------------------------------------
# Target sample
# ------------------------------------------------------------

#df_pca.loc[df_pca["id"].isin(target),"point_category"] = "Target"


# ============================================================
# Functional groups available for widget
# ============================================================

functional_groups = sorted(
    df_pca["functional_group"]
    .dropna()
    .unique()
    .tolist()
)


# ============================================================
# Widget
# ============================================================

pca_functional_widget = widgets.Dropdown(
    options=["All"] + functional_groups,
    value="All",
    description="Group:"
)


# ============================================================
# Point category order
# ============================================================

point_category_order = [
    "Grasses",
    "Herbaceous_legumes",
    "Shrub_Trees",
    "Benchmark",
    "Star grass control",
    "Target"
]


# ============================================================
# Point category colors
# ============================================================

point_category_colors = [
    "#4C78A8",   # Grasses
    "#59A14F",   # Herbaceous legumes
    "#F28E2B",   # Shrub/Trees
    "purple",   # Benchmark
    "black",   # Star grass control
    "red"    # Target
]


# ============================================================
# PCA function
# ============================================================

def plot_pca(functional_group):


    # ========================================================
    # Filter data by functional group
    # ========================================================

    if functional_group == "All":

        data = df_pca.copy()

    else:

        data = df_pca[
            df_pca["functional_group"]
            == functional_group
        ].copy()


    # ========================================================
    # Check number of observations
    # ========================================================

    if len(data) < 3:

        print(
            f"Not enough observations for "
            f"'{functional_group}'. "
            f"Found {len(data)} samples."
        )

        return


    # ========================================================
    # PCA matrix
    # ========================================================

    X = data[pca_vars].copy()


    # ========================================================
    # Replace infinite values with NaN
    # ========================================================

    X.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    # ========================================================
    # Impute missing values with column means
    # ========================================================

    X = X.fillna(
        X.mean()
    )


    # ========================================================
    # Check remaining NaNs
    # ========================================================

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        print(
            "These variables still contain missing values: "
            f"{bad_columns}"
        )

        return


    # ========================================================
    # Check zero variance
    # ========================================================

    zero_variance = [
        column
        for column in pca_vars
        if X[column].nunique() <= 1
    ]

    if zero_variance:

        print(
            "These variables have zero variance for "
            f"'{functional_group}': "
            f"{zero_variance}"
        )

        return


    # ========================================================
    # Standardization
    # ========================================================

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )


    # ========================================================
    # PCA
    # ========================================================

    pca = PCA(
        n_components=2
    )

    principal_components = pca.fit_transform(
        X_scaled
    )


    # ========================================================
    # PCA scores
    # ========================================================

    pca_df = pd.DataFrame(
        principal_components,
        columns=[
            "PC1",
            "PC2"
        ],
        index=data.index
    )


    # ========================================================
    # Keep all columns needed for tooltip
    # ========================================================

    available_tooltip_columns = [
        column
        for column in tooltip_columns
        if column in data.columns
    ]


    # ========================================================
    # Final dataframe
    # ========================================================

    final_df = pd.concat(
        [
            data[
                available_tooltip_columns
            ],
            pca_df
        ],
        axis=1
    )


    # ========================================================
    # PCA loadings
    # ========================================================

    loadings = (
        pca.components_.T
        * np.sqrt(
            pca.explained_variance_
        )
    )


    loading_df = pd.DataFrame(
        loadings,
        columns=[
            "PC1",
            "PC2"
        ],
        index=pca_vars
    ).reset_index()


    loading_df.rename(
        columns={
            "index": "variable"
        },
        inplace=True
    )


    # ========================================================
    # Descriptive names for loading variables
    # ========================================================

    loading_df["variable_label"] = (
        loading_df["variable"]
        .map(
            lambda x: column_labels.get(
                x,
                x.replace(
                    "_",
                    " "
                ).title()
            )
        )
    )


    # ========================================================
    # Scale loading vectors
    # ========================================================

    max_pc = np.max(
        np.abs(
            final_df[
                [
                    "PC1",
                    "PC2"
                ]
            ].values
        )
    )


    max_loading = np.max(
        np.abs(
            loading_df[
                [
                    "PC1",
                    "PC2"
                ]
            ].values
        )
    )


    if max_loading == 0:

        vector_scale = 1

    else:

        vector_scale = (
            max_pc
            / max_loading
            * 0.7
        )


    loading_df["PC1_scaled"] = (
        loading_df["PC1"]
        * vector_scale
    )

    loading_df["PC2_scaled"] = (
        loading_df["PC2"]
        * vector_scale
    )


    # ========================================================
    # Vector dataframe
    # ========================================================

    vector_rows = []

    for _, row in loading_df.iterrows():

        vector_rows.append(
            {
                "x": 0,
                "y": 0,
                "x2": row["PC1_scaled"],
                "y2": row["PC2_scaled"],
                "variable": row["variable_label"]
            }
        )


    vector_df = pd.DataFrame(
        vector_rows
    )


    # ========================================================
    # Tooltip
    # ========================================================

    tooltip = [

        alt.Tooltip(
            column,
            title=column_labels.get(
                column,
                column.replace(
                    "_",
                    " "
                ).title()
            )
        )

        for column in tooltip_columns

        if column in final_df.columns

    ]


    # --------------------------------------------------------
    # Add PCA coordinates
    # --------------------------------------------------------

    tooltip += [

        alt.Tooltip(
            "PC1:Q",
            title="PC1",
            format=".3f"
        ),

        alt.Tooltip(
            "PC2:Q",
            title="PC2",
            format=".3f"
        )

    ]


    # ========================================================
    # PCA points
    # ========================================================

    points = (

        alt.Chart(
            final_df
        )

        .mark_circle(
            size=70
        )

        .encode(


            # ------------------------------------------------
            # X axis
            # ------------------------------------------------

            x=alt.X(
                "PC1:Q",
                title=(
                    f"PC1 "
                    f"({pca.explained_variance_ratio_[0] * 100:.2f}%)"
                )
            ),


            # ------------------------------------------------
            # Y axis
            # ------------------------------------------------

            y=alt.Y(
                "PC2:Q",
                title=(
                    f"PC2 "
                    f"({pca.explained_variance_ratio_[1] * 100:.2f}%)"
                )
            ),


            # ------------------------------------------------
            # Color
            # ------------------------------------------------

            color=alt.Color(

                "point_category:N",

                scale=alt.Scale(

                    domain=point_category_order,

                    range=point_category_colors

                ),

                legend=alt.Legend(
                    title=(
                        "Functional Group / "
                        "Special Samples"
                    )
                )
            ),


            # ------------------------------------------------
            # Tooltip
            # ------------------------------------------------

            tooltip=tooltip

        )
    )


    # ========================================================
    # Loading vectors
    # ========================================================

    vectors = (

        alt.Chart(
            vector_df
        )

        .mark_rule(
            color="red",
            opacity=0.7
        )

        .encode(

            x=alt.X(
                "x:Q"
            ),

            y=alt.Y(
                "y:Q"
            ),

            x2="x2:Q",

            y2="y2:Q"

        )
    )


    # ========================================================
    # Loading labels
    # ========================================================

    labels = (

        alt.Chart(
            vector_df
        )

        .mark_text(
            align="left",
            dx=5,
            dy=-5,
            color="darkred",
            fontWeight="bold",
            fontSize=12
        )

        .encode(

            x=alt.X(
                "x2:Q"
            ),

            y=alt.Y(
                "y2:Q"
            ),

            text=alt.Text(
                "variable:N"
            )

        )
    )


    # ========================================================
    # Final PCA biplot
    # ========================================================

    final_plot = (

        points
        + vectors
        + labels

    ).properties(

        width=850,

        height=600,

        title=(
            f"PCA Biplot - "
            f"{functional_group} "
            f"(n = {len(final_df)})"
        )

    ).configure_axis(

        labelFontSize=14,

        titleFontSize=18

    ).configure_legend(

        titleFontSize=16,

        labelFontSize=14

    ).configure_title(

        fontSize=20

    ).interactive()


    # ========================================================
    # Display
    # ========================================================

    display(
        final_plot
    )


# ============================================================
# Interactive output
# ============================================================

pca_output = widgets.interactive_output(
    plot_pca,
    {
        "functional_group":
            pca_functional_widget
    }
)

# 4.0 Interactive PCA plot

In [5]:
display(widgets.VBox([pca_functional_widget,pca_output]))

# 5.0 Save as interactive HTML

In [10]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy scikit-learn
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import json
import numpy as np
import pandas as pd
import altair as alt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Disable Altair's default 5000-row safety limit, in case df_pca
# (times the number of functional groups) exceeds it.
alt.data_transformers.disable_max_rows()

# ============================================================
# PCA variables
# ============================================================

pca_vars = [
    "dm_percentage",
    "ash_dm",
    "om_percentage",
    "pc_percentage_dm",
    "adf_percentage_dm",
    "ndf_percentage_dm",
    "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h",
    "methane_intensity",
    "tddm"
]

# ============================================================
# Column labels
# ============================================================

column_labels = {
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "point_category": "Plot category",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (% DM)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking"
}

tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group", "point_category",
    "subset", "n_replicates_nutrition", "dm_percentage", "ash_dm",
    "om_percentage", "pc_percentage_dm", "adf_percentage_dm",
    "ndf_percentage_dm", "n_replicates_gas", "ch4_percentage_in_gas_8h",
    "ch4_percentage_in_gas_24h", "methane_intensity", "tddm",
    "ch4_category", "tddm_category", "lmf_category",
    "lmf_category_rank", "quartile", "quartile_rank"
]

# "Target" is left out here on purpose: in the upstream df_pca-building
# code, the line that assigns point_category = "Target" is commented
# out, so no row ever actually takes that value. Keeping it in the
# scale's domain would leave a permanent, empty "Target" legend entry.
point_category_order = [
    "Grasses", "Herbaceous_legumes", "Shrub_Trees",
    "Benchmark", "Star grass control"
]

point_category_colors = [
    "#4C78A8", "#59A14F", "#F28E2B",
    "purple", "black"
]

# ============================================================
# NOTE: this assumes `df_pca` has already been built exactly as
# in the original script (filtered, with point_category assigned, etc.)
# ============================================================

missing_pca_vars = [c for c in pca_vars if c not in df_pca.columns]
if missing_pca_vars:
    raise ValueError(f"The following PCA variables are missing from df_pca: {missing_pca_vars}")

functional_groups = sorted(df_pca["functional_group"].dropna().unique().tolist())
groups_to_run = ["All"] + functional_groups


def compute_pca_for_group(data, group_name):
    """Run PCA on a subset of the data and return (points, vectors), both labeled."""

    if len(data) < 3:
        print(f"Not enough observations for '{group_name}'. Found {len(data)} samples.")
        return None, None

    X = data[pca_vars].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(X.mean())

    if X.isna().any().any():
        bad = X.columns[X.isna().any()].tolist()
        print(f"These variables still contain missing values for '{group_name}': {bad}")
        return None, None

    zero_var = [c for c in pca_vars if X[c].nunique() <= 1]
    if zero_var:
        print(f"These variables have zero variance for '{group_name}': {zero_var}")
        return None, None

    # Standardization
    X_scaled = StandardScaler().fit_transform(X)

    # PCA
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    pc1_pct = pca.explained_variance_ratio_[0] * 100
    pc2_pct = pca.explained_variance_ratio_[1] * 100

    pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"], index=data.index)

    available_tooltip_cols = [c for c in tooltip_columns if c in data.columns]
    final_df = pd.concat([data[available_tooltip_cols], pca_df], axis=1)
    final_df["pca_run"] = group_name
    final_df["axis_title_x"] = f"PC1 ({pc1_pct:.2f}%)"
    final_df["axis_title_y"] = f"PC2 ({pc2_pct:.2f}%)"
    # Percentages are folded into the title too, as a guaranteed fallback
    # in case the JS-based axis-title update (below) can't find the
    # right DOM nodes in a given browser/Vega version.
    final_df["plot_title"] = (
        f"PCA Biplot - {group_name} (n = {len(final_df)})  "
        f"[PC1: {pc1_pct:.2f}%, PC2: {pc2_pct:.2f}%]"
    )

    # PCA loadings
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    loading_df = pd.DataFrame(loadings, columns=["PC1", "PC2"], index=pca_vars).reset_index()
    loading_df.rename(columns={"index": "variable"}, inplace=True)
    loading_df["variable_label"] = loading_df["variable"].map(
        lambda x: column_labels.get(x, x.replace("_", " ").title())
    )

    # Scale loading vectors
    max_pc = np.max(np.abs(final_df[["PC1", "PC2"]].values))
    max_loading = np.max(np.abs(loading_df[["PC1", "PC2"]].values))
    vector_scale = 1 if max_loading == 0 else (max_pc / max_loading * 0.7)

    loading_df["PC1_scaled"] = loading_df["PC1"] * vector_scale
    loading_df["PC2_scaled"] = loading_df["PC2"] * vector_scale

    vector_rows = []
    for _, row in loading_df.iterrows():
        vector_rows.append({
            "x": 0, "y": 0,
            "x2": row["PC1_scaled"], "y2": row["PC2_scaled"],
            "variable": row["variable_label"],
            "pca_run": group_name
        })
    vector_df = pd.DataFrame(vector_rows)

    return final_df, vector_df


# ============================================================
# Pre-compute PCA for every group
# ============================================================

all_points, all_vectors = [], []

for group in groups_to_run:
    subset = df_pca.copy() if group == "All" else df_pca[df_pca["functional_group"] == group].copy()
    pts, vecs = compute_pca_for_group(subset, group)
    if pts is not None:
        all_points.append(pts)
        all_vectors.append(vecs)

points_all = pd.concat(all_points, ignore_index=True)
vectors_all = pd.concat(all_vectors, ignore_index=True)

# Per-group axis title lookup, embedded into the HTML for the JS-based
# axis title update (best effort — see note in the JS section below).
axis_titles_lookup = (
    points_all
    .drop_duplicates("pca_run")
    .set_index("pca_run")[["axis_title_x", "axis_title_y"]]
    .rename(columns={"axis_title_x": "x", "axis_title_y": "y"})
    .to_dict(orient="index")
)

# ============================================================
# Interactive dropdown selector (rendered at the top of the chart)
# ============================================================

group_dropdown = alt.binding_select(
    options=groups_to_run,
    name="Functional group: "
)

group_param = alt.param(
    name="sel_group",
    value="All",
    bind=group_dropdown
)

group_filter_expr = "datum.pca_run == sel_group"

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

tooltip = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in tooltip_columns
    if col in points_all.columns
] + [
    alt.Tooltip("PC1:Q", title="PC1", format=".3f"),
    alt.Tooltip("PC2:Q", title="PC2", format=".3f"),
]

# ------------------------------------------------------------
# Title row (includes the PC1/PC2 percentages as a guaranteed
# fallback, in addition to the best-effort axis title update)
# ------------------------------------------------------------

title_chart = (
    alt.Chart(points_all)
    .mark_text(align="center", fontSize=18, dy=0)
    .encode(text="plot_title:N")
    .transform_filter(group_filter_expr)
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# PCA points — base layer (all non-matched points, by category)
# and highlight layer (only matched points, one shape per id)
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=70)
    .transform_filter(group_filter_expr)
    .transform_filter(f"!({is_match_expr})")
    .encode(
        x=alt.X("PC1:Q", title="PC1"),
        y=alt.Y("PC2:Q", title="PC2"),
        color=alt.Color(
            "point_category:N",
            scale=alt.Scale(domain=point_category_order, range=point_category_colors),
            legend=alt.Legend(title="Functional Group", symbolSize=70)
        ),
        tooltip=tooltip
    )
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=70, stroke="black", strokeWidth=1.5)
    .transform_filter(group_filter_expr)
    .transform_filter(is_match_expr)
    .encode(
        x=alt.X("PC1:Q"),
        y=alt.Y("PC2:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=70, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
    .add_params(search_param)
)

points_chart = base_layer + highlight_layer

# ------------------------------------------------------------
# Loading vectors
# ------------------------------------------------------------

vectors_chart = (
    alt.Chart(vectors_all)
    .mark_rule(color="red", opacity=0.7)
    .encode(x="x:Q", y="y:Q", x2="x2:Q", y2="y2:Q")
    .transform_filter(group_filter_expr)
)

# ------------------------------------------------------------
# Loading labels
# ------------------------------------------------------------

labels_chart = (
    alt.Chart(vectors_all)
    .mark_text(align="left", dx=5, dy=-5, color="darkred", fontWeight="bold", fontSize=12)
    .encode(x="x2:Q", y="y2:Q", text="variable:N")
    .transform_filter(group_filter_expr)
)

# ------------------------------------------------------------
# Final PCA biplot — dropdown on top, then title, then the plot
# ------------------------------------------------------------

biplot = (
    (points_chart + vectors_chart + labels_chart)
    .properties(width=850, height=600)
)

final_plot = (
    alt.vconcat(title_chart, biplot)
    .add_params(group_param, search_param)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    .interactive()
)

# ============================================================
# Save as interactive HTML with the dropdown forced above the chart
# ============================================================

chart_spec = final_plot.to_json(indent=None)
axis_titles_json = json.dumps(axis_titles_lookup)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>PCA Biplot</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>

  <script type="text/javascript">
    const spec = {chart_spec};
    const axisTitles = {axis_titles_json};

    // Vega renders at 96 CSS pixels per inch by default. Scaling the
    // canvas by (300 / 96) makes the exported pixel dimensions match
    // a 300 dpi print at the chart's original physical size, for
    // every format below (PNG, JPG, TIFF, and PDF alike).
    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg"
    }}).then(function(result) {{

      const view = result.view;

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "pca_plot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "pca_plot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "pca_plot.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("pca_plot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // Explicitly hide the "Highlighted accession(s)" legend while the
      // search box is empty (Vega-Lite already leaves it empty of
      // entries in that case, but this makes it fully invisible).
      function toggleHighlightedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');
        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      // BEST EFFORT: update the actual x/y axis titles to show the
      // PC1/PC2 percentages for the currently selected group. This
      // inspects the rendered SVG directly (Vega-Lite has no built-in
      // way to bind an axis title to a live parameter), so it depends
      // on internal DOM structure that could vary between Vega
      // versions. The percentages are ALSO shown in the chart's main
      // title above, which does not depend on this and always works.
      function updateAxisTitles(group) {{
        const info = axisTitles[group];
        if (!info) return;
        const container = view.container();
        if (!container) return;

        const axisGroups = container.querySelectorAll('[aria-roledescription="axis"]');
        axisGroups.forEach(function(g) {{
          // The axis title is typically a direct <text> child of the
          // axis group, while tick labels live inside a nested <g>.
          const titleEl = g.querySelector(":scope > text");
          if (!titleEl) return;
          const transform = titleEl.getAttribute("transform") || "";
          const isRotated = transform.indexOf("rotate") !== -1;
          titleEl.textContent = isRotated ? info.y : info.x;
        }});
      }}

      function refreshDynamicLabels() {{
        const group = view.signal("sel_group") || "All";
        updateAxisTitles(group);
      }}

      toggleHighlightedAccessionsLegend(view.signal("search_id"));
      view.addSignalListener("search_id", function(name, value) {{
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 50);
      }});

      // Re-apply axis titles once on load, and whenever the dropdown
      // selection changes.
      setTimeout(refreshDynamicLabels, 50);
      view.addSignalListener("sel_group", function() {{
        setTimeout(refreshDynamicLabels, 50);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("interactive_pca.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: interactive_pca.html")

Done: interactive_pca.html
